# Notebook 3: Cross-Engine Queries

We run five queries that combine Cypher graph traversal with DuckDB analytical queries, each illustrating a different integration pattern.

| Query | Pattern | Description |
|-------|---------|-------------|
| 1 | Graph filter -> DuckDB aggregate | Cypher finds Manhattan zones; DuckDB aggregates trip stats for those zones |
| 2 | Graph routing -> DuckDB enrich | Cypher finds the top 10 busiest routes; DuckDB adds avg trip duration from raw data |
| 3 | DuckDB scan -> graph context | DuckDB finds the top 5 pickup zones; Cypher returns zone name, borough and connectivity |
| 4 | DuckDB cross-file join | A single `duckdb.query()` reads both Parquet files and compares yellow vs. green fares |
| 5 | Summary | Cypher counts graph elements; DuckDB counts raw Parquet rows -- side by side |

## Prerequisites

Set the following environment variables before starting Jupyter:

```shell
export NEO4J_URI="bolt://localhost:7687"
export NEO4J_USERNAME="neo4j"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="neo4j"
```

Run Notebook 2 first to build the graph.

## 1. Install dependencies

In [1]:
%pip install neo4j==6.1.0 --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and configuration

In [2]:
import os

from neo4j import GraphDatabase

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

RAW_DIR = os.path.expanduser("~/neo4j-duckdb-taxi/data/raw")

YELLOW_PARQUET = os.path.join(RAW_DIR, "yellow_tripdata_2024-01.parquet")
GREEN_PARQUET  = os.path.join(RAW_DIR, "green_tripdata_2024-01.parquet")

print("Credentials set.")

Credentials set.


## 3. Connect to Neo4j

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 4. Query 1 - graph filter, DuckDB aggregation

In [5]:
# Query 1: Zone lookup via Cypher, aggregation via DuckDB
#
# Cypher finds all Zone locationIds in Manhattan.
# DuckDB reads the yellow Parquet and aggregates trip stats
# for those zones -- the graph supplies the filter,
# DuckDB does the analytical heavy lifting.

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("""
        MATCH (z:Zone {borough: 'Manhattan'})
        RETURN z.locationId AS locationId
        ORDER BY z.locationId
    """)
    manhattan_ids = [r["locationId"] for r in result]

print(f"Manhattan zones found by Cypher: {len(manhattan_ids)}")

id_list = ", ".join(str(i) for i in manhattan_ids)

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(f"""
        CALL duckdb.query(
            'SELECT
                PULocationID                 AS location_id,
                COUNT(*)                     AS trip_count,
                ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
                ROUND(AVG(fare_amount), 2)   AS avg_fare_usd
             FROM read_parquet("{YELLOW_PARQUET}")
             WHERE PULocationID IN ({id_list})
             GROUP BY PULocationID
             ORDER BY trip_count DESC
             LIMIT 10'
        )
        YIELD row
        RETURN row.location_id        AS locationId,
               row.trip_count         AS tripCount,
               row.avg_distance_miles AS avgDistanceMiles,
               row.avg_fare_usd       AS avgFareUsd
    """)
    rows = result.data()

print("\nTop 10 Manhattan pickup zones (yellow cab, Jan 2024):")
print(f"{'locationId':>12}  {'tripCount':>10}  {'avgDistMiles':>13}  {'avgFareUSD':>11}")
print("-" * 55)
for r in rows:
    print(f"{r['locationId']:>12}  {r['tripCount']:>10,}  {r['avgDistanceMiles']:>13.2f}  {r['avgFareUsd']:>11.2f}")

Manhattan zones found by Cypher: 69

Top 10 Manhattan pickup zones (yellow cab, Jan 2024):
  locationId   tripCount   avgDistMiles   avgFareUSD
-------------------------------------------------------
         161     143,471           2.56        15.21
         237     142,708           1.70        12.18
         236     136,465           1.85        12.71
         162     106,717           2.23        14.79
         230     106,324           2.91        17.54
         186     104,523           2.27        15.79
         142     104,080           2.09        13.43
         239      88,474           2.26        13.45
         163      85,692           2.50        15.01
         170      83,064           2.20        14.74


## 5. Query 2 - graph routing, DuckDB enrichment

In [6]:
# Query 2: Graph-first, then DuckDB enrichment
#
# Cypher finds the top 10 busiest yellow cab routes by
# relationship count stored in the graph.
# DuckDB then reads the raw Parquet for those exact zone pairs
# to return average trip duration -- a property not pre-aggregated
# on the relationship.

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("""
        MATCH (pu:Zone)-[t:YELLOW_TRIP]->(do:Zone)
        RETURN pu.locationId AS puId,
               pu.zone       AS puZone,
               do.locationId AS doId,
               do.zone       AS doZone,
               t.count       AS tripCount
        ORDER BY t.count DESC
        LIMIT 10
    """)
    top_routes = result.data()

print("Top 10 routes from the graph:")
for r in top_routes:
    print(f"  {r['puZone']} -> {r['doZone']}  ({r['tripCount']:,} trips)")

# Build a VALUES clause for DuckDB
values = ", ".join(
    f"({r['puId']}, {r['doId']})"
    for r in top_routes
)

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(f"""
        CALL duckdb.query(
            'WITH pairs(pu_id, do_id) AS (
                VALUES {values}
            )
            SELECT
                t.PULocationID AS pu_id,
                t.DOLocationID AS do_id,
                ROUND(AVG((epoch_ms(tpep_dropoff_datetime) - epoch_ms(tpep_pickup_datetime)) / 60000.0), 2) AS avg_duration_minutes
            FROM read_parquet("{YELLOW_PARQUET}") t
            JOIN pairs p ON t.PULocationID = p.pu_id AND t.DOLocationID = p.do_id
            GROUP BY t.PULocationID, t.DOLocationID
            ORDER BY avg_duration_minutes DESC'
        )
        YIELD row
        RETURN row.pu_id                AS puId,
               row.do_id                AS doId,
               row.avg_duration_minutes AS avgDurationMinutes
    """)
    durations = {(r["puId"], r["doId"]): r["avgDurationMinutes"] for r in result}

print("\nTop 10 routes enriched with avg trip duration from raw Parquet:")
print(f"{'Pickup zone':<30}  {'Dropoff zone':<30}  {'Trips':>8}  {'Avg min':>8}")
print("-" * 85)
for r in top_routes:
    dur = durations.get((r["puId"], r["doId"]), None)
    dur_str = f"{dur:.2f}" if dur is not None else "n/a"
    print(f"{r['puZone']:<30}  {r['doZone']:<30}  {r['tripCount']:>8,}  {dur_str:>8}")

Top 10 routes from the graph:
  Upper East Side South -> Upper East Side North  (753 trips)
  Upper East Side North -> Upper East Side South  (652 trips)
  Upper East Side North -> Upper East Side North  (503 trips)
  Upper East Side South -> Upper East Side South  (489 trips)
  Midtown Center -> Upper East Side South  (335 trips)
  Lincoln Square East -> Upper West Side South  (327 trips)
  Upper West Side South -> Lincoln Square East  (327 trips)
  Upper East Side South -> Midtown Center  (317 trips)
  Upper West Side South -> Upper West Side North  (280 trips)
  Midtown Center -> Upper East Side North  (279 trips)

Top 10 routes enriched with avg trip duration from raw Parquet:
Pickup zone                     Dropoff zone                       Trips   Avg min
-------------------------------------------------------------------------------------
Upper East Side South           Upper East Side North                753      7.83
Upper East Side North           Upper East Side South     

## 6. Query 3 - DuckDB scan, graph context

In [7]:
# Query 3: DuckDB-first, then graph context
#
# DuckDB reads the raw Parquet to find the top 5 pickup zones
# by trip volume.
# Cypher then pulls each zone's name, borough and connectivity
# from the graph -- how many distinct zones does each connect to?

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(f"""
        CALL duckdb.query(
            'SELECT
                PULocationID AS location_id,
                COUNT(*)     AS trip_count
             FROM read_parquet("{YELLOW_PARQUET}")
             WHERE PULocationID IS NOT NULL
             GROUP BY PULocationID
             ORDER BY trip_count DESC
             LIMIT 5'
        )
        YIELD row
        RETURN row.location_id AS locationId,
               row.trip_count  AS tripCount
    """)
    top_pickups = result.data()

print("Top 5 pickup zones from DuckDB (raw Parquet):")
for r in top_pickups:
    print(f"  locationId {r['locationId']:>4}  ->  {r['tripCount']:>8,} trips")

top_ids = [r["locationId"] for r in top_pickups]

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("""
        UNWIND $ids AS locId
        MATCH (z:Zone {locationId: locId})
        OPTIONAL MATCH (z)-[:YELLOW_TRIP|GREEN_TRIP]->(neighbor:Zone)
        RETURN z.locationId   AS locationId,
               z.zone         AS zone,
               z.borough      AS borough,
               count(DISTINCT neighbor) AS connectedZones
        ORDER BY connectedZones DESC
    """, ids=top_ids)
    graph_context = result.data()

print("\nGraph context for top pickup zones:")
print(f"{'locationId':>12}  {'Zone':<35}  {'Borough':<15}  {'Connected zones':>15}")
print("-" * 85)
for r in graph_context:
    print(f"{r['locationId']:>12}  {r['zone']:<35}  {r['borough']:<15}  {r['connectedZones']:>15}")

Top 5 pickup zones from DuckDB (raw Parquet):
  locationId  132  ->   145,240 trips
  locationId  161  ->   143,471 trips
  locationId  237  ->   142,708 trips
  locationId  236  ->   136,465 trips
  locationId  162  ->   106,717 trips

Graph context for top pickup zones:
  locationId  Zone                                 Borough          Connected zones
-------------------------------------------------------------------------------------
         132  JFK Airport                          Queens                       233
         162  Midtown East                         Manhattan                    113
         161  Midtown Center                       Manhattan                    109
         237  Upper East Side South                Manhattan                    107
         236  Upper East Side North                Manhattan                    107


## 7. Query 4 - cross-file DuckDB join

In [8]:
# Query 4: Yellow vs. green comparison
#
# A single duckdb.query() call reads both Parquet files and
# joins them on shared zone pairs to compare avg fare and
# avg distance for yellow vs. green cabs.
# DuckDB handles the cross-file join entirely inside Neo4j.

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(f"""
        CALL duckdb.query(
            'WITH yellow AS (
                SELECT
                    PULocationID,
                    DOLocationID,
                    COUNT(*)                     AS trip_count,
                    ROUND(AVG(fare_amount), 2)   AS avg_fare,
                    ROUND(AVG(trip_distance), 2) AS avg_distance
                FROM read_parquet("{YELLOW_PARQUET}")
                WHERE PULocationID IS NOT NULL
                  AND DOLocationID IS NOT NULL
                GROUP BY PULocationID, DOLocationID
            ),
            green AS (
                SELECT
                    PULocationID,
                    DOLocationID,
                    COUNT(*)                     AS trip_count,
                    ROUND(AVG(fare_amount), 2)   AS avg_fare,
                    ROUND(AVG(trip_distance), 2) AS avg_distance
                FROM read_parquet("{GREEN_PARQUET}")
                WHERE PULocationID IS NOT NULL
                  AND DOLocationID IS NOT NULL
                GROUP BY PULocationID, DOLocationID
            )
            SELECT
                y.PULocationID          AS pu_id,
                y.DOLocationID          AS do_id,
                y.trip_count            AS yellow_trips,
                y.avg_fare              AS yellow_avg_fare,
                y.avg_distance          AS yellow_avg_distance,
                g.trip_count            AS green_trips,
                g.avg_fare              AS green_avg_fare,
                g.avg_distance          AS green_avg_distance
            FROM yellow y
            JOIN green g
              ON y.PULocationID = g.PULocationID
             AND y.DOLocationID = g.DOLocationID
            ORDER BY y.trip_count DESC
            LIMIT 10'
        )
        YIELD row
        RETURN row.pu_id               AS puId,
               row.do_id               AS doId,
               row.yellow_trips        AS yellowTrips,
               row.yellow_avg_fare     AS yellowAvgFare,
               row.yellow_avg_distance AS yellowAvgDistance,
               row.green_trips         AS greenTrips,
               row.green_avg_fare      AS greenAvgFare,
               row.green_avg_distance  AS greenAvgDistance
    """)
    comparison = result.data()

print("Top 10 shared zone pairs -- yellow vs. green (Jan 2024):")
print(f"{'PU':>5}  {'DO':>5}  {'Yel trips':>10}  {'Yel fare':>9}  {'Yel dist':>9}  {'Grn trips':>10}  {'Grn fare':>9}  {'Grn dist':>9}")
print("-" * 80)
for r in comparison:
    print(
        f"{r['puId']:>5}  {r['doId']:>5}  "
        f"{r['yellowTrips']:>10,}  {r['yellowAvgFare']:>9.2f}  {r['yellowAvgDistance']:>9.2f}  "
        f"{r['greenTrips']:>10,}  {r['greenAvgFare']:>9.2f}  {r['greenAvgDistance']:>9.2f}"
    )

Top 10 shared zone pairs -- yellow vs. green (Jan 2024):
   PU     DO   Yel trips   Yel fare   Yel dist   Grn trips   Grn fare   Grn dist
--------------------------------------------------------------------------------
  236    237      19,402       9.17       1.04          30      12.10       1.50
  236    236      15,955       6.86       0.62          49       8.51       0.70
  264    264       7,381      21.53       2.78          86      20.17       0.77
  132    132       7,252      26.61       1.57          19      41.79       1.71
  236    161       7,115      14.14       1.92           8      17.61       2.39
  238    239       7,007       7.06       0.78           1      15.50       0.00
  236    141       6,930       9.56       1.11          17      11.85       1.66
  236    239       6,808      10.11       1.43          11      12.93       1.94
  263    236       6,574       6.67       0.67           2       5.80       0.50
  236    238       6,385      10.04       1.40      

## 8. Query 5 - summary

In [9]:
# Query 5: Summary -- both engines together
#
# Cypher counts what's in the graph: Zone nodes and trip
# relationships.
# duckdb.query() counts what's in the raw Parquet files.
# Both results printed side by side to show the two engines
# contributing to a single summary.

with driver.session(database=NEO4J_DATABASE) as session:
    graph_counts = session.run("""
        MATCH (z:Zone)               WITH count(z) AS zones
        MATCH ()-[y:YELLOW_TRIP]->() WITH zones, count(y) AS yellow
        MATCH ()-[g:GREEN_TRIP]->()  RETURN zones, yellow, count(g) AS green
    """).single()

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(f"""
        CALL duckdb.query(
            'SELECT
                (SELECT COUNT(*) FROM read_parquet("{YELLOW_PARQUET}")) AS yellow_rows,
                (SELECT COUNT(*) FROM read_parquet("{GREEN_PARQUET}"))  AS green_rows'
        )
        YIELD row
        RETURN row.yellow_rows AS yellowRows,
               row.green_rows  AS greenRows
    """)
    raw_counts = result.single()

print("=" * 45)
print("Dataset summary")
print("=" * 45)
print()
print("Neo4j graph (Cypher)")
print(f"  Zone nodes:       {graph_counts['zones']:>10,}")
print(f"  YELLOW_TRIP rels: {graph_counts['yellow']:>10,}")
print(f"  GREEN_TRIP rels:  {graph_counts['green']:>10,}")
print()
print("Raw Parquet files (DuckDB)")
print(f"  Yellow cab rows:  {raw_counts['yellowRows']:>10,}")
print(f"  Green cab rows:   {raw_counts['greenRows']:>10,}")

Dataset summary

Neo4j graph (Cypher)
  Zone nodes:              265
  YELLOW_TRIP rels:      7,031
  GREEN_TRIP rels:       5,314

Raw Parquet files (DuckDB)
  Yellow cab rows:   2,964,624
  Green cab rows:       56,551


## 9. Teardown

In [10]:
driver.close()
print("Connection closed.")

Connection closed.
